# Projeto MedIR-RAG — Milestone 2 (Recuperação Densa com FAISS)
## Comparação Lexical (BM25 da M1) vs Recuperação Densa Zero-Shot (Bi-Encoder + FAISS)

Versão focada e limpa que integra exclusivamente o que foi pedido:
1. **Carregamento e Split Fixo (Train/Test):** Reutilização da coleção baseada em snippets e fixação do conjunto de teste (seed 13) para garantir comparabilidade com a Milestone 1.
2. **Baseline BM25:** Avaliação com a tua baseline da Milestone 1.
3. **Recuperação Densa com FAISS:** Exatamente a classe `FAISSDenseRetriever` e o Bi-Encoder (`BAAI/bge-small-en-v1.5`) do exemplo do professor.
4. **Tabela Comparativa Simples:** Comparação direta de métricas (`NDCG@5`, `NDCG@10`, `MRR@10`, `MAP`, `Recall@100`) calculada em Python/NumPy (compatível com Windows, sem erros de compilação).

### 1. Instalação e Imports

In [1]:
%pip install -q faiss-cpu sentence-transformers rank_bm25 pandas numpy torch tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import faiss
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

# Fixação de seeds para reprodutibilidade
SEED = 13
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo em utilização: {device}")

c:\Users\diogu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dispositivo em utilização: cpu


### 2. Carregamento dos Dados e Preparação das Estruturas
Carregamos o ficheiro `BioASQ-trainingDataset2b.json` da Milestone 1 e montamos os DataFrames `df_corpus` (`docno`, `text`) e `df_test_queries` (`qid`, `query`).

In [5]:
# Caminho para o ficheiro BioASQ
DATASET_PATH = "BioASQ2-2014-Training-2b.json"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)["questions"]

# 1. Construir corpus agregando snippets por PMID
corpus_dict = {}
for q in raw_data:
    for snip in q.get("snippets", []):
        doc_url = snip.get("document", "")
        doc_id = doc_url.split("/")[-1] if "/" in doc_url else doc_url.split("=")[-1]
        text = snip.get("text", "").strip()
        if doc_id:
            if doc_id in corpus_dict:
                corpus_dict[doc_id] += " " + text
            else:
                corpus_dict[doc_id] = text

df_corpus = pd.DataFrame([
    {"docno": str(doc_id), "text": text}
    for doc_id, text in corpus_dict.items()
])
print(f"Total de passagens no corpus: {len(df_corpus)}")

# 2. Construir queries e qrels
all_queries = []
qrels = {}

for q in raw_data:
    qid = str(q["id"])
    query_text = q["body"].strip()
    gold_docs = [
        d.split("/")[-1] if "/" in d else d.split("=")[-1]
        for d in q.get("documents", [])
    ]
    valid_gold = [str(d) for d in gold_docs if str(d) in corpus_dict]
    if len(valid_gold) > 0:
        qrels[qid] = {d: 1 for d in valid_gold}
        all_queries.append({"qid": qid, "query": query_text})

df_all_queries = pd.DataFrame(all_queries)

# Seleção das 300 questões de teste com seed=13 (idêntico à Milestone 1)
df_test_queries = df_all_queries.sample(n=min(300, len(df_all_queries)), random_state=SEED).reset_index(drop=True)
test_qids = set(df_test_queries["qid"].tolist())
qrels_test = {qid: docs for qid, docs in qrels.items() if qid in test_qids}

print(f"Total de queries de teste selecionadas: {len(df_test_queries)}")

Total de passagens no corpus: 3539
Total de queries de teste selecionadas: 300


### 3. Função de Avaliação de Métricas (Python / NumPy Nnativo)
Cálculo direto de NDCG@5, NDCG@10, MRR@10, MAP e Recall@100 sem depender de compilação C/C++ externa.

In [6]:
def compute_ndcg_at_k(retrieved_docs, gold_docs, k):
    dcg = 0.0
    for i, doc in enumerate(retrieved_docs[:k]):
        if doc in gold_docs:
            dcg += 1.0 / np.log2(i + 2)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(gold_docs), k)))
    return dcg / idcg if idcg > 0 else 0.0

def compute_mrr_at_k(retrieved_docs, gold_docs, k):
    for i, doc in enumerate(retrieved_docs[:k]):
        if doc in gold_docs:
            return 1.0 / (i + 1)
    return 0.0

def compute_map(retrieved_docs, gold_docs):
    hits, sum_prec = 0, 0.0
    for i, doc in enumerate(retrieved_docs):
        if doc in gold_docs:
            hits += 1
            sum_prec += hits / (i + 1)
    return sum_prec / len(gold_docs) if gold_docs else 0.0

def compute_recall_at_k(retrieved_docs, gold_docs, k):
    hits = sum(1 for doc in retrieved_docs[:k] if doc in gold_docs)
    return hits / len(gold_docs) if gold_docs else 0.0

def evaluate_run(run_dict, qrels_dict):
    ndcg5, ndcg10, mrr10, map_scores, recall100 = [], [], [], [], []
    for qid, gold_map in qrels_dict.items():
        if qid not in run_dict:
            continue
        gold_docs = set(gold_map.keys())
        ranked_docs = sorted(run_dict[qid].keys(), key=lambda d: run_dict[qid][d], reverse=True)
        
        ndcg5.append(compute_ndcg_at_k(ranked_docs, gold_docs, 5))
        ndcg10.append(compute_ndcg_at_k(ranked_docs, gold_docs, 10))
        mrr10.append(compute_mrr_at_k(ranked_docs, gold_docs, 10))
        map_scores.append(compute_map(ranked_docs, gold_docs))
        recall100.append(compute_recall_at_k(ranked_docs, gold_docs, 100))
        
    return {
        "NDCG@5": round(float(np.mean(ndcg5)), 4),
        "NDCG@10": round(float(np.mean(ndcg10)), 4),
        "MRR@10": round(float(np.mean(mrr10)), 4),
        "MAP": round(float(np.mean(map_scores)), 4),
        "Recall@100": round(float(np.mean(recall100)), 4)
    }

results_comparison = {}

### 4. Baseline Lexical (BM25 da Milestone 1)
Executamos a busca lexical com os parâmetros otimizados ($k_1=0.6, b=0.75$).

In [7]:
print("A indexar com BM25...")
tokenized_corpus = [doc.lower().split() for doc in df_corpus["text"]]
bm25 = BM25Okapi(tokenized_corpus, k1=0.6, b=0.75)
doc_ids = df_corpus["docno"].tolist()

print("A avaliar BM25 no conjunto de teste...")
run_bm25 = {}
for _, row in tqdm(df_test_queries.iterrows(), total=len(df_test_queries), desc="BM25"):
    qid = str(row["qid"])
    q_tokens = row["query"].lower().split()
    scores = bm25.get_scores(q_tokens)
    
    top_idx = np.argpartition(scores, -100)[-100:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    run_bm25[qid] = {doc_ids[idx]: float(scores[idx]) for idx in top_idx}

results_comparison["BM25 (Milestone 1)"] = evaluate_run(run_bm25, qrels_test)
print("Resultados BM25:", results_comparison["BM25 (Milestone 1)"])

A indexar com BM25...
A avaliar BM25 no conjunto de teste...


BM25: 100%|██████████| 300/300 [00:02<00:00, 140.27it/s]

Resultados BM25: {'NDCG@5': 0.5417, 'NDCG@10': 0.4984, 'MRR@10': 0.7292, 'MAP': 0.3844, 'Recall@100': 0.6215}


### 5. Recuperação Densa com FAISS (Exemplo do Professor)
Classe `FAISSDenseRetriever` e Bi-Encoder conforme apresentado pelo professor.

In [8]:
class FAISSDenseRetriever:
    """
    Classe para gestão do índice vetorial FAISS e codificação com Bi-Encoder.
    """
    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f" -> A carregar Bi-Encoder: {model_name} (Dispositivo: {self.device})")
        self.model = SentenceTransformer(model_name, device=self.device)
        self.index = None
        self.docno_list = []

    def build_index(self, df_corpus, batch_size=128):
        print(" -> A gerar embeddings do corpus e a criar índice FAISS...")
        texts = df_corpus["text"].tolist()
        self.docno_list = df_corpus["docno"].tolist()
        
        # Embeddings normalizados para similaridade por cosseno com Inner Product
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        dimension = embeddings.shape[1]
        
        # Construção do Índice FAISS (IndexFlatIP: Produto Interno)
        self.index = faiss.IndexFlatIP(dimension)
        self.index.add(np.array(embeddings, dtype=np.float32))
        print(f" -> Índice FAISS construído com {self.index.ntotal} vetores.")

    def search(self, df_queries, top_k=100):
        print(" -> A pesquisar queries no índice FAISS...")
        queries = df_queries["query"].tolist()
        qids = df_queries["qid"].tolist()
        
        q_embeddings = self.model.encode(
            queries,
            show_progress_bar=True,
            normalize_embeddings=True
        )
        scores, indices = self.index.search(np.array(q_embeddings, dtype=np.float32), top_k)
        
        run_dict = {}
        for i, qid in enumerate(qids):
            run_dict[str(qid)] = {}
            for doc_idx, score in zip(indices[i], scores[i]):
                if doc_idx != -1:
                    run_dict[str(qid)][self.docno_list[doc_idx]] = float(score)
        return run_dict

In [9]:
# 1. Instanciar e construir o índice FAISS com o modelo do professor
faiss_retriever = FAISSDenseRetriever(model_name="BAAI/bge-small-en-v1.5")
faiss_retriever.build_index(df_corpus, batch_size=128)

# 2. Realizar a busca sobre as queries de teste
run_faiss_dense = faiss_retriever.search(df_test_queries, top_k=100)

# 3. Avaliar recuperação densa
results_comparison["Dense FAISS (bge-small-en-v1.5)"] = evaluate_run(run_faiss_dense, qrels_test)
print("Resultados Dense FAISS:", results_comparison["Dense FAISS (bge-small-en-v1.5)"])

 -> A carregar Bi-Encoder: BAAI/bge-small-en-v1.5 (Dispositivo: cpu)


c:\Users\diogu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\diogu\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\diogu\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\fi

 -> A gerar embeddings do corpus e a criar índice FAISS...


Batches: 100%|██████████| 28/28 [01:07<00:00,  2.41s/it]


 -> Índice FAISS construído com 3539 vetores.
 -> A pesquisar queries no índice FAISS...


Batches: 100%|██████████| 10/10 [00:01<00:00,  9.03it/s]

Resultados Dense FAISS: {'NDCG@5': 0.8387, 'NDCG@10': 0.8166, 'MRR@10': 0.9285, 'MAP': 0.7521, 'Recall@100': 0.9321}


### 6. Comparação Direta: BM25 vs FAISS Dense

In [10]:
df_final_comparison = pd.DataFrame(results_comparison).T
print("=======================================================================")
print("               COMPARAÇÃO: LEXICAL (BM25) vs DENSO (FAISS)             ")
print("=======================================================================")
display(df_final_comparison)

# Guardar resultados em CSV
df_final_comparison.to_csv("dense_vs_bm25_results.csv")
print("Execução concluída com sucesso!")

               COMPARAÇÃO: LEXICAL (BM25) vs DENSO (FAISS)             


,NDCG@5,NDCG@10,MRR@10,MAP,Recall@100
BM25 (Milestone 1),0.5417,0.4984,0.7292,0.3844,0.6215
Dense FAISS (bge-small-en-v1.5),0.8387,0.8166,0.9285,0.7521,0.9321


Execução concluída com sucesso!
